In [10]:
import numpy as np

ROWS, COLS = 6, 7
EMPTY, MAX_PLAYER, MIN_PLAYER = 0, 1, 2

class Connect4:
    def __init__(self):
        self.board = np.zeros((ROWS, COLS), dtype=int)
        self.current_player = MAX_PLAYER

    def actions(self, board):
        return [c for c in range(COLS) if board[0][c] == EMPTY]

    def drop_piece(self, board, col, player):
        b = board.copy()
        for row in range(ROWS - 1, -1, -1):
            if b[row][col] == EMPTY:
                b[row][col] = player
                break
        return b

    def check_winner(self, board, player):
        # Horizontal
        for r in range(ROWS):
            for c in range(COLS - 3):
                if all(board[r][c+i] == player for i in range(4)):
                    return True
        # Vertical
        for r in range(ROWS - 3):
            for c in range(COLS):
                if all(board[r+i][c] == player for i in range(4)):
                    return True
        # Diagonal /
        for r in range(3, ROWS):
            for c in range(COLS - 3):
                if all(board[r-i][c+i] == player for i in range(4)):
                    return True
        # Diagonal \
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                if all(board[r+i][c+i] == player for i in range(4)):
                    return True
        return False

    def is_terminal(self, board):
        return (self.check_winner(board, MAX_PLAYER) or
                self.check_winner(board, MIN_PLAYER) or
                len(self.actions(board)) == 0)

    def utility(self, board):
        if self.check_winner(board, MAX_PLAYER):
            return 1000
        elif self.check_winner(board, MIN_PLAYER):
            return -1000
        return 0

In [11]:
def minimax(game, board, depth, is_maximizing):
    if depth == 0 or game.is_terminal(board):
        return game.utility(board)

    if is_maximizing:
        best = -float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MAX_PLAYER)
            best = max(best, minimax(game, new_board, depth - 1, False))
        return best
    else:
        best = float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MIN_PLAYER)
            best = min(best, minimax(game, new_board, depth - 1, True))
        return best


def get_best_move(board, depth):
    game = Connect4()
    best_score = -float('inf')
    best_col = game.actions(board)[0]

    for col in game.actions(board):
        new_board = game.drop_piece(board, col, MAX_PLAYER)
        score = minimax(game, new_board, depth - 1, False)
        if score > best_score:
            best_score = score
            best_col = col

    return best_col

In [12]:
# contador global de nodos
nodes_minimax = 0
nodes_alphabeta = 0

def minimax_counted(game, board, depth, is_maximizing):
    global nodes_minimax
    nodes_minimax += 1

    if depth == 0 or game.is_terminal(board):
        return game.utility(board)

    if is_maximizing:
        best = -float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MAX_PLAYER)
            best = max(best, minimax_counted(game, new_board, depth - 1, False))
        return best
    else:
        best = float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MIN_PLAYER)
            best = min(best, minimax_counted(game, new_board, depth - 1, True))
        return best


def alphabeta(game, board, depth, alpha, beta, is_maximizing):
    global nodes_alphabeta
    nodes_alphabeta += 1

    if depth == 0 or game.is_terminal(board):
        return game.utility(board)

    if is_maximizing:
        best = -float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MAX_PLAYER)
            best = max(best, alphabeta(game, new_board, depth - 1, alpha, beta, False))
            alpha = max(alpha, best)  # actualiza alpha
            if beta <= alpha:
                break  # poda beta
        return best
    else:
        best = float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MIN_PLAYER)
            best = min(best, alphabeta(game, new_board, depth - 1, alpha, beta, True))
            beta = min(beta, best)  # actualiza beta
            if beta <= alpha:
                break  # poda alfa
        return best


def get_best_move_ab(board, depth):
    game = Connect4()
    best_score = -float('inf')
    best_col = game.actions(board)[0]

    for col in game.actions(board):
        new_board = game.drop_piece(board, col, MAX_PLAYER)
        score = alphabeta(game, new_board, depth - 1, -float('inf'), float('inf'), False)
        if score > best_score:
            best_score = score
            best_col = col

    return best_col

In [13]:
# Demostración comparativa de algoritmos
game = Connect4()

# estado de prueba: algunas fichas ya puestas
test_board = game.board.copy()
test_board = game.drop_piece(test_board, 3, MAX_PLAYER)
test_board = game.drop_piece(test_board, 3, MIN_PLAYER)
test_board = game.drop_piece(test_board, 2, MAX_PLAYER)
test_board = game.drop_piece(test_board, 4, MIN_PLAYER)

DEPTH = 4

nodes_minimax = 0
move_mm = get_best_move(test_board, DEPTH)  # usa minimax original
minimax_counted(game, test_board, DEPTH, True)  # solo para contar

nodes_alphabeta = 0
move_ab = get_best_move_ab(test_board, DEPTH)

print(f"Minimax  -> Movimiento: columna {move_mm} | Nodos visitados: {nodes_minimax}")
print(f"AlphaBeta-> Movimiento: columna {move_ab} | Nodos visitados: {nodes_alphabeta}")
print(f"Reducción: {100 - (nodes_alphabeta / nodes_minimax * 100):.1f}%")

Minimax  -> Movimiento: columna 0 | Nodos visitados: 2717
AlphaBeta-> Movimiento: columna 0 | Nodos visitados: 501
Reducción: 81.6%


In [14]:
def evaluate(board):
    score = 0

    # control del centro (columna 3) - posicion estrategica clave
    center_col = [board[r][3] for r in range(ROWS)]
    score += center_col.count(MAX_PLAYER) * 3
    score -= center_col.count(MIN_PLAYER) * 3

    # evalúa ventanas de 4 en todas las direcciones
    def score_window(window, player):
        opp = MIN_PLAYER if player == MAX_PLAYER else MAX_PLAYER
        s = 0
        if window.count(player) == 4:
            s += 1000
        elif window.count(player) == 3 and window.count(EMPTY) == 1:  # 3 fichas + espacio libre
            s += 5
        elif window.count(player) == 2 and window.count(EMPTY) == 2:  # 2 fichas + 2 espacios
            s += 2
        if window.count(opp) == 3 and window.count(EMPTY) == 1:
            s -= 4  # bloquear amenaza del oponente
        return s

    def scan_windows(player):
        s = 0
        # horizontal
        for r in range(ROWS):
            for c in range(COLS - 3):
                window = list(board[r, c:c+4])
                s += score_window(window, player)
        # vertical
        for r in range(ROWS - 3):
            for c in range(COLS):
                window = [board[r+i][c] for i in range(4)]
                s += score_window(window, player)
        # diagonal /
        for r in range(3, ROWS):
            for c in range(COLS - 3):
                window = [board[r-i][c+i] for i in range(4)]
                s += score_window(window, player)
        # diagonal \
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                window = [board[r+i][c+i] for i in range(4)]
                s += score_window(window, player)
        return s

    score += scan_windows(MAX_PLAYER)
    score -= scan_windows(MIN_PLAYER)
    return score

In [15]:
# AlphaBeta con heurística en nodos hojas

def alphabeta_h(game, board, depth, alpha, beta, is_maximizing):
    if game.is_terminal(board):
        return game.utility(board)
    if depth == 0:
        return evaluate(board)  # heurística en hojas

    if is_maximizing:
        best = -float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MAX_PLAYER)
            best = max(best, alphabeta_h(game, new_board, depth - 1, alpha, beta, False))
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    else:
        best = float('inf')
        for col in game.actions(board):
            new_board = game.drop_piece(board, col, MIN_PLAYER)
            best = min(best, alphabeta_h(game, new_board, depth - 1, alpha, beta, True))
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best


def get_best_move_h(board, depth):
    game = Connect4()
    best_score = -float('inf')
    best_col = game.actions(board)[0]

    for col in game.actions(board):
        new_board = game.drop_piece(board, col, MAX_PLAYER)
        score = alphabeta_h(game, new_board, depth - 1, -float('inf'), float('inf'), False)
        if score > best_score:
            best_score = score
            best_col = col

    return best_col

In [16]:
import random

def print_board(board):
    print("\n 0 1 2 3 4 5 6")
    print("-" * 15)
    for row in board:
        print("|" + "|".join(["." if c == EMPTY else "X" if c == MAX_PLAYER else "O" for c in row]) + "|")
    print("-" * 15)


def play_vs_random():
    game = Connect4()
    board = game.board.copy()
    print("\nIA (X) vs Aleatorio (O)")

    while True:
        print_board(board)

        # turno IA
        col = get_best_move_h(board, 6)
        board = game.drop_piece(board, col, MAX_PLAYER)
        print(f"IA juega columna: {col}")
        if game.is_terminal(board):
            break

        # turno aleatorio
        col = random.choice(game.actions(board))
        board = game.drop_piece(board, col, MIN_PLAYER)
        print(f"Aleatorio juega columna: {col}")
        if game.is_terminal(board):
            break

    print_board(board)
    if game.check_winner(board, MAX_PLAYER):
        print("Gana la IA!")
    elif game.check_winner(board, MIN_PLAYER):
        print("Gana el Aleatorio!")
    else:
        print("Empate!")


def play_vs_human():
    game = Connect4()
    board = game.board.copy()
    print("\nHumano (X) vs IA (O)")
    print("Tú eres X, la IA es O. Ingresa el número de columna (0-6).")

    # humano es MAX_PLAYER, IA es MIN_PLAYER
    while True:
        print_board(board)

        # turno humano
        while True:
            try:
                col = int(input("Tu columna: "))
                if col in game.actions(board):
                    break
                print("Columna inválida, intenta de nuevo.")
            except ValueError:
                print("Ingresa un número.")

        board = game.drop_piece(board, col, MAX_PLAYER)
        if game.is_terminal(board):
            break

        # turno IA
        col = get_best_move_h(board, 6)
        board = game.drop_piece(board, col, MIN_PLAYER)
        print(f"IA juega columna: {col}")
        if game.is_terminal(board):
            break

    print_board(board)
    if game.check_winner(board, MAX_PLAYER):
        print("Ganaste!")
    elif game.check_winner(board, MIN_PLAYER):
        print("Gana la IA!")
    else:
        print("Empate!")


In [17]:

play_vs_random()


IA (X) vs Aleatorio (O)

 0 1 2 3 4 5 6
---------------
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
---------------
IA juega columna: 3
Aleatorio juega columna: 5

 0 1 2 3 4 5 6
---------------
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|X|.|O|.|
---------------
IA juega columna: 3
Aleatorio juega columna: 1

 0 1 2 3 4 5 6
---------------
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|X|.|.|.|
|.|O|.|X|.|O|.|
---------------
IA juega columna: 1
Aleatorio juega columna: 5

 0 1 2 3 4 5 6
---------------
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|X|.|X|.|O|.|
|.|O|.|X|.|O|.|
---------------
IA juega columna: 1
Aleatorio juega columna: 6

 0 1 2 3 4 5 6
---------------
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|.|.|.|.|.|.|
|.|X|.|.|.|.|.|
|.|X|.|X|.|O|.|
|.|O|.|X|.|O|O|
---------------
IA juega columna: 5
Aleatorio juega columna: 3

 0 1 2 3 4 5 6
----

In [18]:

#play_vs_human()